In [ ]:
import base64
import os
from email.mime.text import MIMEText

import pandas as pd
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build


In [ ]:
# Email configuration
sender_email = "utahmug@gmail.com"

contacts = pd.read_csv("contacts.csv")
contacts_mug = contacts[contacts["Label"].str.contains("MUG List", na=False)]
receiver_emails = contacts_mug["Email"].tolist()
# receiver_emails = ["chris.day@wfrc.utah.gov"]

# subject = "New Blog Post Alert"
subject = "RSG is Hiring"
message_body = """
<p>Hello Model Users,</p>
<p>Please take a look at the most recent blog post for details about a job opening at RSG. <a href="https://utahmug.org/rsg-job-opening/">https://utahmug.org/rsg-job-opening/</a></p>
<p><i>If you do not want to receive email updates, please let Chris Day know at chris.day@wfrc.utah.gov and he will take you off the list of recipients.</i></p>
"""

# Set up Gmail API
SCOPES = ["https://www.googleapis.com/auth/gmail.send"]
creds = None

if os.path.exists("token.json"):
    creds = Credentials.from_authorized_user_file("token.json", SCOPES)

if not creds or not creds.valid:
    flow = InstalledAppFlow.from_client_secrets_file(
        "client_secrets.json", SCOPES, redirect_uri="http://localhost/8080"
    )  # maybe 8000
    creds = flow.run_local_server(port=8080)

# Save the credentials for the next run
with open("token.json", "w") as token:
    token.write(creds.to_json())

# Create the Gmail service
service = build("gmail", "v1", credentials=creds)

# Send the email to multiple recipients
for receiver_email in receiver_emails:
    # Create the email message using MIMEText
    message = MIMEText(message_body, "html")
    message["to"] = receiver_email
    message["from"] = sender_email
    message["subject"] = subject

    # Encode the message as base64
    email_message = base64.urlsafe_b64encode(message.as_bytes()).decode("utf-8")

    try:
        message = (
            service.users()
            .messages()
            .send(userId="me", body={"raw": email_message})
            .execute()
        )
        print(f"Email sent successfully to {receiver_email}.")
    except Exception as e:
        print(f"An error occurred while sending the email to {receiver_email}: {e}")

if os.path.exists("token.json"):
    os.remove("token.json")
    print("token.json file has been deleted.")


Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=248540739780-oacuh3cavr74ltot2k3rmbthbbrj3ce3.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8080%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.send&state=dB0PCW0Vf6qv4Gg2sCwbzeUQlt91oy&access_type=offline
Email sent successfully to awilding@utah.gov.
Email sent successfully to andy.li@wfrc.utah.gov.
Email sent successfully to austin.feula@wcg.us.
Email sent successfully to ben.swanson@wcg.us.
Email sent successfully to bert.granberg@wfrc.utah.gov.
Email sent successfully to B.Liu@fehrandpeers.com.
Email sent successfully to bill.hereth@wfrc.utah.gov.
Email sent successfully to Bishoy.Kelleny@bentley.com.
Email sent successfully to bbrady@summitcounty.org.
Email sent successfully to brent.turley@transpogroup.com.
Email sent successfully to Camille.Alexander@hdrinc.com.
Email sent successfully to cmiller@summitcounty.org.
Email sent suc